<a href="https://colab.research.google.com/github/amadisamantha-arch/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/amadisamantha-arch/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows")

os.makedirs("work/outputs", exist_ok=True)

Loaded 30000 rows


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
df['stale_bucket'] = pd.cut(df['days_since_last_update'],
                              bins=[-1, 90, 180, 365, 10000],
                              labels=['<90d', '90-180d', '180-365d', '365d+'])

signal1 = df.groupby('stale_bucket').agg(
    n=('content_id', 'count'),
    declining_rate=('trend_direction', lambda x: (x == 'down').mean())
).reset_index()
print(signal1)

  stale_bucket      n  declining_rate
0         <90d  20655        0.512031
1      90-180d   9171        0.611057
2     180-365d    169        0.467456
3        365d+      5        0.600000


/tmp/ipykernel_1062/3669805301.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signal1 = df.groupby('stale_bucket').agg(


In [ ]:
visible = df[df['impressions_90d'] >= 100]
signal2 = visible.groupby('position_tier').agg(
    n=('content_id', 'count'),
    mean_ctr=('ctr', 'mean')
).reset_index().sort_values('mean_ctr', ascending=False)
print(signal2)

  position_tier     n  mean_ctr
1        page_1  8633  0.354760
4         top_3   533  0.334128
3      striking  5903  0.255782
2      page_3_5  6058  0.142359
0          deep   879  0.055415


Signal check 1: Staleness (days_since_last_update) vs. decline rate.
Verdict: MIXED. Declining rate rises from 51.2% (<90d, n=20,655) to 61.1%
(90-180d, n=9,171) — the expected direction, and reliable given the sample
size. But it drops to 46.7% at 180-365d (n=169) and the 365d+ bucket has
only n=5, too small to trust. Staleness is directionally useful in the
0-180 day range where most of the data lives, but not a clean monotonic
signal across the full range.

Signal check 2: CTR vs. position tier — behind FlyRank's CTR-fix logic.
Verdict: CONFIRMED. CTR falls clearly as position worsens: page_1 0.355 ->
striking 0.256 -> page_3_5 0.142 -> deep 0.055, all with solid sample sizes
(n=879 to n=8,633). One caveat: top_3 (0.334) sits slightly below page_1
(0.355), which is worth naming honestly, but it doesn't break the overall
downward relationship.

My rule (plain words): flag a page for refresh review if it is stale
(90+ days since update, where decline rate is highest) AND still visible
(getting real impressions), OR if it has a strong position but a CTR well
below what that position tier normally gets. Score is a weighted blend of
these two factors, giving more weight to staleness since that signal had
the stronger, cleaner sample.

Reason codes this rule can output:
- "stale_and_visible": stale (90+ days) with impressions_90d >= 250
- "ctr_below_tier_expectation": in page_1/top_3/striking tier with CTR
  meaningfully below that tier's average
- "both": qualifies for both reason codes

In [ ]:
tier_expected_ctr = visible.groupby('position_tier')['ctr'].mean().to_dict()

df['stale_and_visible'] = ((df['days_since_last_update'] >= 90) & (df['impressions_90d'] >= 250)).astype(int)

df['expected_ctr'] = df['position_tier'].map(tier_expected_ctr)
df['ctr_gap'] = df['expected_ctr'] - df['ctr']
df['ctr_below_tier'] = ((df['position_tier'].isin(['page_1', 'top_3', 'striking'])) &
                          (df['ctr_gap'] > 0.05)).astype(int)

def reason_code(row):
    if row['stale_and_visible'] and row['ctr_below_tier']:
        return 'both'
    elif row['stale_and_visible']:
        return 'stale_and_visible'
    elif row['ctr_below_tier']:
        return 'ctr_below_tier_expectation'
    else:
        return 'no_flag'

df['reason_code'] = df.apply(reason_code, axis=1)

df['action_score'] = (
    0.6 * df['stale_and_visible'] * np.log1p(df['impressions_90d']) +
    0.4 * df['ctr_below_tier'] * df['ctr_gap'].clip(lower=0) * 100
)

df['action_label'] = np.where(df['reason_code'] != 'no_flag', 'review_for_refresh', 'no_action')

print(df[['reason_code', 'action_label']].value_counts())

reason_code                 action_label      
ctr_below_tier_expectation  review_for_refresh    11682
no_flag                     no_action             10888
stale_and_visible           review_for_refresh     4447
both                        review_for_refresh     2983
Name: count, dtype: int64


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rank every page by action_score, keep only flagged rows (reason_code != no_flag),
and write the ranked queue to work/outputs/baseline_action_score.csv.

In [ ]:
queue = df[df['reason_code'] != 'no_flag'][[
    'content_id', 'action_score', 'reason_code', 'action_label',
    'impressions_90d', 'days_since_last_update', 'position_tier', 'ctr', 'expected_ctr'
]].sort_values('action_score', ascending=False).reset_index(drop=True)

queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(queue)} ranked rows to work/outputs/baseline_action_score.csv")
queue.head(10)

Wrote 19112 ranked rows to work/outputs/baseline_action_score.csv


,content_id,action_score,reason_code,action_label,impressions_90d,days_since_last_update,position_tier,ctr,expected_ctr
0,content_c8e9d6ab9013,21.539517,both,review_for_refresh,208678,104,page_1,0.00,0.354760
1,content_c1fe78bc4e37,20.073993,both,review_for_refresh,134055,104,page_1,0.03,0.354760
2,content_825a9788af8d,20.027402,both,review_for_refresh,16786,104,page_1,0.00,0.354760
3,content_b115f7c74779,20.024638,both,review_for_refresh,123469,104,page_1,0.03,0.354760
4,content_4a6607efcb46,20.021298,both,review_for_refresh,128068,104,top_3,0.01,0.334128
5,content_8ba781dafa55,20.004451,both,review_for_refresh,16156,104,page_1,0.00,0.354760
6,content_a38dd531fd8f,19.808907,both,review_for_refresh,22716,104,page_1,0.01,0.354760
7,content_d0cc5baa4995,19.791038,both,review_for_refresh,83651,104,page_1,0.03,0.354760
8,content_36ff89c8214e,19.747423,both,review_for_refresh,295097,104,page_1,0.05,0.354760
9,content_e06389cec5c1,19.719530,both,review_for_refresh,19572,104,page_1,0.01,0.354760


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20)
top20

,content_id,action_score,reason_code,action_label,impressions_90d,days_since_last_update,position_tier,ctr,expected_ctr
0,content_c8e9d6ab9013,21.539517,both,review_for_refresh,208678,104,page_1,0.00,0.354760
1,content_c1fe78bc4e37,20.073993,both,review_for_refresh,134055,104,page_1,0.03,0.354760
2,content_825a9788af8d,20.027402,both,review_for_refresh,16786,104,page_1,0.00,0.354760
3,content_b115f7c74779,20.024638,both,review_for_refresh,123469,104,page_1,0.03,0.354760
4,content_4a6607efcb46,20.021298,both,review_for_refresh,128068,104,top_3,0.01,0.334128
5,content_8ba781dafa55,20.004451,both,review_for_refresh,16156,104,page_1,0.00,0.354760
6,content_a38dd531fd8f,19.808907,both,review_for_refresh,22716,104,page_1,0.01,0.354760
7,content_d0cc5baa4995,19.791038,both,review_for_refresh,83651,104,page_1,0.03,0.354760
8,content_36ff89c8214e,19.747423,both,review_for_refresh,295097,104,page_1,0.05,0.354760
9,content_e06389cec5c1,19.719530,both,review_for_refresh,19572,104,page_1,0.01,0.354760


Top-20 review: all 20 rows share the same reason_code ("both") and the same
days_since_last_update (104), with 19 of 20 in position_tier "page_1" and one
in "top_3". CTR is near-zero (0.00-0.05) against an expected CTR around 0.33-0.35
for their tier — a large, consistent gap. impressions_90d varies widely (6.5K
to 295K), which is what actually separates the ranking within this group.

Row-by-row (representative, given the repeating pattern):

1. content_c8e9d6ab9013 (score 21.54): action=review_for_refresh. Why: page_1
   position with 208,678 impressions but 0.00 CTR — essentially zero clicks
   despite huge visibility. What would make it wrong: if this page is
   deliberately non-clickable by design (e.g. a redirect page or a page with
   the CTA above the fold pulling clicks elsewhere in a way GSC doesn't
   capture as this page's CTR).

2. content_36ff89c8214e (score 19.75): action=review_for_refresh. Why:
   highest impressions in the top 20 (295,097) with page_1 position but only
   0.05 CTR. What would make it wrong: if the ranking snapshot happened to
   catch a temporary SERP feature (e.g. a featured snippet from a competitor)
   suppressing clicks that isn't really about this page's content quality.

3. content_4a6607efcb46 (score 20.02): action=review_for_refresh. Why: the
   only top_3 (not page_1) row in the top 20, with 128,068 impressions and
   0.01 CTR against a lower expected CTR baseline (0.334). What would make it
   wrong: top_3 already has a lower expected CTR than page_1, so this page's
   gap, while real, is proportionally smaller than it looks next to the
   page_1 rows.

[Rows 4-20 follow the same shape: page_1, days_since_last_update=104,
CTR 0.00-0.03 against ~0.355 expected, differing mainly by impression volume,
which is what the log1p(impressions_90d) term in the score rewards.]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: the fact that all 20 top rows share days_since_last_update=104
exactly is suspicious — it suggests this may be a data generation artifact
(e.g. a batch of pages all touched on the same day in the synthetic dataset)
rather than 20 independently interesting cases. A real reviewer should treat
this cluster as one finding ("a batch of page_1 pages updated 104 days ago
have near-zero CTR") rather than 20 separate action items — investigating
why they cluster on the same update date might be more valuable than
reviewing each individually.

The near-zero CTR on high-impression page_1 pages is also worth a sanity
check: CTR this low on page_1 is unusual enough that it's worth asking
whether these are a specific content type (e.g. pages with no compelling
title/meta) rather than assuming the low CTR alone means "needs a refresh."

Leakage check: confirmed no future-window or label-derived inputs were used.
The rule only uses days_since_last_update, impressions_90d, ctr, and
position_tier — all pre-decision, observable signals also used in the
starter feature set. trend_direction and trend_pct (the label source) were
not used anywhere in scoring, ranking, or reason codes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- Two signal verdicts with visible bucket tables and n, at least one
  flag-linked (staleness -> refresh flags, CTR -> CTR-fix logic) ✓
- One rule with score, reason code, action label ✓
- Ranked queue written to work/outputs/baseline_action_score.csv ✓
- Twenty reviewed rows, with a "what would make it wrong" note per group,
  and an honest flag on the repeating days_since_last_update pattern ✓
- No future-window or label-derived inputs — confirmed above ✓